<a href="https://colab.research.google.com/github/govind0429/AI_LAB/blob/main/WEEK7_UNIFICATION_1BM24CS106.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import re

def parse(s):
    s = s.replace(" ", "")
    tokens = re.findall(r'[A-Za-z0-9_]+|[(),]', s)
    pos = 0

    def term():
        nonlocal pos
        name = tokens[pos]
        pos += 1

        if pos < len(tokens) and tokens[pos] == '(':
            pos += 1
            args = []
            while tokens[pos] != ')':
                args.append(term())
                if tokens[pos] == ',':
                    pos += 1
            pos += 1
            return (name, *args)

        return name

    result = term()
    if pos != len(tokens):
        raise ValueError("Invalid expression")
    return result


def is_var(x):
    return isinstance(x, str) and x in {'x', 'y', 'z', 'u', 'v', 'w'}


def substitute(x, sub):
    if is_var(x) and x in sub:
        return substitute(sub[x], sub)
    if isinstance(x, tuple):
        return tuple([x[0]] + [substitute(a, sub) for a in x[1:]])
    return x


def occurs(v, x, sub):
    x = substitute(x, sub)
    if v == x:
        return True
    if isinstance(x, tuple):
        return any(occurs(v, a, sub) for a in x[1:])
    return False


def unify(x, y, sub):
    x = substitute(x, sub)
    y = substitute(y, sub)

    if x == y:
        return sub

    if is_var(x):
        if occurs(x, y, sub):
            return None
        sub[x] = y
        return sub

    if is_var(y):
        return unify(y, x, sub)

    if isinstance(x, tuple) and isinstance(y, tuple):
        if x[0] != y[0] or len(x) != len(y):
            return None

        for a, b in zip(x[1:], y[1:]):
            sub = unify(a, b, sub)
            if sub is None:
                return None
        return sub

    return None


def display(x):
    if isinstance(x, tuple):
        return x[0] + "(" + ", ".join(display(a) for a in x[1:]) + ")"
    return str(x)


n = int(input("Enter number of test cases: "))

for i in range(n):
    print("\nTest Case", i + 1)

    s1 = input("Enter first expression: ")
    s2 = input("Enter second expression: ")

    try:
        e1 = parse(s1)
        e2 = parse(s2)

        result = unify(e1, e2, {})

        if result is None:
            print("Unification Failed")
        else:
            print("Unification Successful")
            print("Substitutions:")
            for k, v in result.items():
                print(k, "=", display(substitute(v, result)))

            print("Unified Expression:",
                  display(substitute(e1, result)))

    except (ValueError, IndexError):
        print("Invalid input")

Enter number of test cases: 2

Test Case 1
Enter first expression: P(A,B,B)
Enter second expression: P(x,y,z)
Unification Successful
Substitutions:
x = A
y = B
z = B
Unified Expression: P(A, B, B)

Test Case 2
Enter first expression: Q(y,G(A,B))
Enter second expression: Q(G(x,x),y)
Unification Failed
